# Streaming VAE Anomaly Detection — GPU Demo

Clones the repo and runs `demo.py` end to end on a free Colab GPU: offline
warmup training + online streaming anomaly detection on a real ERA5 slice.

Runtime menu → Change runtime type → GPU, then run all cells.

**Setup:** this is a private repo, so before running you need a GitHub token as a Colab secret — key icon in the left sidebar -> new secret named `GITHUB_TOKEN`, value = a token from [github.com/settings/tokens](https://github.com/settings/tokens) (read-only `repo` access is enough), then toggle "Notebook access" on.

### Choose which experiment to run

In [ ]:
ANOM = "contextual"    # point | group | contextual
ARCH = "mlp_cyclic"    # mlp | mlp_cyclic | lstm | transformer
BEST = False            # True -> run the finetuned config instead of baseline

print(f"ANOM={ANOM!r}  ARCH={ARCH!r}  BEST={BEST}")


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
import os

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Private repo: add a GitHub token as a Colab secret first --
    # key icon in the left sidebar -> Secrets -> new secret named GITHUB_TOKEN,
    # value = a token from github.com/settings/tokens (read-only "repo" access is enough)
    # -> toggle "Notebook access" on.
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    if not os.path.exists("repo"):
        !git clone $clone_url repo
    %cd repo
elif not os.path.exists("run_regression.py"):
    # Already inside a local checkout (e.g. running from notebooks/) --
    # move to the repo root instead of cloning a redundant nested copy.
    %cd ..

!pip install -q -r requirements.txt


### Run `demo.py`

Same script you'd run from a terminal — `python demo.py --anom ... --arch ... [--best]`.
See `demo.py --help` for the full option list.

In [ ]:
_best_flag = "--best" if BEST else ""
!python demo.py --anom {ANOM} --arch {ARCH} {_best_flag}


### Inspect the run

In [ ]:
import glob
import yaml

_tuning = "finetuned" if BEST else "baseline"
_anom_title = {"point": "Point", "group": "Group", "contextual": "Contextual"}[ANOM]
_arch_dir = {
    "mlp": "MLP", "mlp_cyc": "MLP_Cyclic", "mlp_cyclic": "MLP_Cyclic",
    "lstm": "LSTM", "tr": "Transformer", "tf": "Transformer",
    "transformer": "Transformer", "transtormer": "Transformer",
}[ARCH]
_config_path = glob.glob(f"experiments/{_tuning}/{_anom_title}_{_arch_dir}/*_config_standalone.yaml")[0]
with open(_config_path) as f:
    _cfg = yaml.safe_load(f)

RUN_DIR = _cfg["common"]["logging"]["run_dir"]
TEST_PATH = _cfg["data"]["test_path"]

# The METRICS / THRESHOLD TUNING block at the end of run.log is the run's own summary
with open(f"{RUN_DIR}/run.log") as f:
    lines = f.readlines()
print("".join(lines[-40:]))


### Injected anomalies vs. the real signal (one feature)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(TEST_PATH, parse_dates=["valid_time"], index_col="valid_time")
feature = "t2m"

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(df.index, df[feature], lw=0.8, label=feature)
anom = df[df["is_anomaly"] == 1]
ax.scatter(anom.index, anom[feature], color="red", s=10, label="injected anomaly", zorder=3)
ax.legend()
ax.set_title(f"Real ERA5 test series ({ANOM} scenario) — {feature}")
plt.show()
